# Redis 1

# Lab: Redis

In [ ]:
import redis

import json

import pprint

from datetime import timedelta

import time

import numpy as np


## Connect to the redis database; redis the container name is used for the host name; TCP port is 6379; databases are given a integer number instead of a name

In [ ]:
cities_db = redis.Redis(host='redis', port=6379, db=0)

## Flush (delete all) from the cities database in case there are key / value pairs left from prior runs

In [ ]:
cities_db.flushdb()

## Example of key / value pairs where both the key and the value are simple strings; the logic for key / value pairs is basically the same as a Python dictionary:  if the key does not exist, create a new "slot" with the key / value pair, if the key already exists, overwrite the value

In [ ]:
cities_db.set("Berkeley", "California")
cities_db.set("Portland", "Oregon")
cities_db.set("Seattle", "Washington")
cities_db.set("Dallas", "Texas")
cities_db.set("Nashville", "Tennessee")
cities_db.set("Chicago", "Illinois")
cities_db.set("Miami", "Florida")

In [ ]:
cities_db.keys()

In [ ]:
cities_db.get("Berkeley")

In [ ]:
for key in cities_db.keys():
    print("key:", key, "value:", cities_db.get(key))

In [ ]:
cities_db.set("Berkeley", "CA")
cities_db.set("Portland", "OR")
cities_db.set("Seattle", "WA")
cities_db.set("Dallas", "TX")
cities_db.set("Nashville", "TN")
cities_db.set("Chicago", "IL")
cities_db.set("Miami", "FL")

In [ ]:
for key in cities_db.keys():
    print("key:", key, "value:", cities_db.get(key))

## If we want to set a lot of values at the same time, we can use a feature called pipelining, which is more efficient than setting them one at a time; not until we execute the pipeline do the changes get made;

In [ ]:
cities_pipe = cities_db.pipeline()

cities_list = [["Berkeley", "California"], 
               ["Portland", "Oregon"],
               ["Seattle", "Washington"],
               ["Dallas", "Texas"],
               ["Nashville", "Tennessee"],
               ["Chicago", "Illinois"],
               ["Miami", "Florida"]
              ]

for (city, state) in cities_list:
    cities_pipe.set(city, state)


In [ ]:
for key in cities_db.keys():
    print("key:", key, "value:", cities_db.get(key))

In [ ]:
cities_pipe.execute()

In [ ]:
for key in cities_db.keys():
    print("key:", key, "value:", cities_db.get(key))

## We can put an expirey on a key / value pair;  once the key / value pair expires, it is deleted; there are methods to specify time in milliseconds, seconds, minutes, specific date, time, etc.; set New York for 10 seconds, watch it disappear

In [ ]:
cities_db.setex("New York", timedelta(seconds=10), "NY")

In [ ]:
cities_db.get("New York")

In [ ]:
# wait 10 seconds and see it disappear
cities_db.get("New York")

## We are been using simple strings, which are not that useful;  JSON values are much more useful and typical;  create a new database 1 (flush it to clean out from prior runs); load the temp_stores_nested.json using the store_id as the key and the JSON as a string as the value

In [ ]:
stores_db = redis.Redis(host='redis', port=6379, db=1)

stores_db.flushdb()

stores_pipe = stores_db.pipeline()

In [ ]:
f = open("temp_stores_nested.json")
file_json = json.load(f)
f.close()

stores_json_list = file_json["stores"]

In [ ]:
for store in stores_json_list:
    stores_pipe.set(store["store_id"], json.dumps(store))
    
stores_pipe.execute()

In [ ]:
# print only the first 90 characters of the JSON object, since they are large

for key in stores_db.keys():
    print("key:", key, "value:", stores_db.get(key)[:90])

## Where redis really shines is that it is a scale out, in memory solution;  memory is backed to disk at periodic intervals, which can be changed in the config file;  one great example would be a leader board for a multi-player online game; we will simulate a multi-player game with random scores 

In [ ]:
games_db = redis.Redis(host='redis', port=6379, db=2)

games_db.flushdb()

game_id = 3874297973

## Run the Jupyter Notebook redis_2 to get the set values (actually you can make several copies of redis_2 and run several at the same time!)

## Infine loop - to stop: menu bar => Kernet => Interrupt

In [ ]:
scores = {}

base_score = 0

while (True):
    
    base_score += 50
    
    scores["Alice"] = base_score + np.random.randint(16,20)
    scores["Bob"] = base_score + np.random.randint(16,20)
    scores["Mary"] = base_score + np.random.randint(11,15)
    scores["Jeff"] = base_score + np.random.randint(11,15)
    scores["Roberta"] = base_score + np.random.randint(6,10)
    scores["Trevor"] = base_score + np.random.randint(6,10)
    scores["Wanda"] = base_score + np.random.randint(1,5)
    scores["Walter"] = base_score + np.random.randint(1,5)
    
    games_db.set(game_id, json.dumps(scores))
    
    time.sleep(1.0)
    

## You try it

## Create database 3 for sales; load the file temp_sales_nested.json into the sales database with the receipt as the key and the JSON as the value

## Create database 4 for customers; load the file temp_customers_nested.json into the customers database with the customer_id as the key and the JSON as the value